Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import os
import shutil

# Step 1: Ensure 'data' directory exists
if not os.path.exists("data"):
    os.makedirs("data")

# Step 2: Run the SQL query and get the results as a DataFrame
query = """

WITH

/* ===================== BASE ===================== */
base AS (
    SELECT
        canonical_ticket_id,
        ticket_group,
        ticket_state,
        phone_number,
        ticket_created_time,
        resolved_time,
        CASE
            WHEN overall_last_agent_type_for_sub_ticket = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
                THEN coalesce(overall_last_agent_name_for_sub_ticket, last_obcs_l1_agent_name)
            ELSE last_obcs_l1_agent_name
        END AS agent_name
    FROM corp_analytics.cx_sub_ticket_data_model_tbl
    WHERE ticket_created_dt >= date_format(current_date - interval '30' day, '%Y%m%d')
      AND overall_last_agent_type_for_sub_ticket = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
),

/* ===================== ME AUDITS ===================== */
me_audits AS (
    SELECT *
    FROM (
        SELECT
            CAST(SUBSTR(claim_last_date_trail_new, 1, 10) AS DATE) AS resolved_date,
            me_name AS agent_name,
            ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY claim_last_date_trail_new DESC
            ) AS rk
        FROM corp_analytics.ss_gi_claims_all_tbl x
        LEFT JOIN (
            SELECT claim_id, MAX(done_by) AS me_name
            FROM (
                SELECT
                    claim_id,
                    done_by,
                    DENSE_RANK() OVER (
                        PARTITION BY claim_id
                        ORDER BY created_at DESC
                    ) AS rk
                FROM gi_claims.claim_audit_trail
                WHERE from_status = 'PENDING_DATA_ENTRY_ME'
            ) t
            WHERE rk = 1
            GROUP BY claim_id
        ) me
        ON me.claim_id = x.claim_id
        WHERE year_week_crd NOT IN ('Null','null')
    )
    WHERE rk = 1
      AND resolved_date >= CURRENT_DATE - INTERVAL '90' DAY
      AND agent_name IS NOT NULL
)

/* ===================== FINAL QUERY ===================== */
SELECT 
    CASE 
        WHEN day_of_week(resolved_date) = 1 
        THEN date_add('day', 2, resolved_date) 
        ELSE date_add('day', 1, resolved_date) 
    END AS Audit_Date,
    resolved_date,
    agent_name
FROM (
    SELECT DISTINCT 
        resolved_date, 
        ticket_id, 
        group_name,
        agent_name
    FROM (

        /* ===================== ORIGINAL QUERY 3 ===================== */

        SELECT 
            hit.agent_name,
            hit.ticket_id,
            hit.group_name,
            DATE(CAST(split_part(REPLACE(hit.resolved_date,'T',' '),'.',1) AS timestamp)) AS resolved_date
        FROM omnitrix.hi_tickets_v1 hit  
        LEFT JOIN omnitrix.freshdesk_ticket_gi_report fd 
            ON fd.ticket_id = hit.ticket_id
        WHERE hit.group_name IN (
                'Senior Callback',
                'Claims Support',
                'Rejection Calling',
                'Claims De-escalation',
                'Claims Support Emails'
              )
        AND DATE(
                COALESCE(
                    CAST(split_part(REPLACE(hit.resolved_date,'T',' '),'.',1) AS timestamp), 
                    CAST(split_part(REPLACE(closed_date,'T',' '),'.',1) AS timestamp)
                )
            ) >= current_date - interval '35' day
        AND (
                (hit.status IN ('Resolved','Closed') 
                    AND hit.group_name IN (
                        'Senior Callback',
                        'Claims Support',
                        'Rejection Calling',
                        'Claims De-escalation'
                    )
                )
             OR (hit.status = 'Resolved' 
                    AND hit.group_name = 'Claims Support Emails'
                )
            )
        AND fd.claims_type NOT IN ('Not Connected')

        UNION

        SELECT 
            hit.agent_name,
            hit.ticket_id,
            hit.group_name,
            DATE(CAST(split_part(REPLACE(hit.resolved_date,'T',' '),'.',1) AS timestamp)) AS resolved_date
        FROM omnitrix.hi_tickets_v1 hit  
        LEFT JOIN omnitrix.freshdesk_ticket_gi_report fd 
            ON fd.ticket_id = hit.ticket_id
        WHERE hit.group_name IN (
                'BIMA Bharosa Portal',
                'manager.experience',
                'Grievance Team'
              )
        AND DATE(
                CAST(split_part(REPLACE(hit.resolved_date,'T',' '),'.',1) AS timestamp)
            ) >= current_date - interval '35' day
        AND hit.status IN ('Resolved','Closed')

        /* ===================== BASE ADDED ===================== */
        UNION

        SELECT
            b.agent_name,
            CAST(b.canonical_ticket_id AS varchar) AS ticket_id,
            'Claims Support' AS group_name,
            DATE(CAST(b.resolved_time AS timestamp)) AS resolved_date
        FROM base b
        WHERE b.resolved_time IS NOT NULL
          AND DATE(CAST(b.resolved_time AS timestamp)) >= CURRENT_DATE - INTERVAL '35' DAY

        /* ===================== ME AUDITS ADDED ===================== */
        UNION

        SELECT
            agent_name,
            CAST(NULL AS varchar) AS ticket_id,
            'ME Audits' AS group_name,
            resolved_date
        FROM me_audits

    )
)
ORDER BY 2

"""
df = get_sql_df(query)

# Step 3: Connect to the Google Sheet
gsheet_url = "https://docs.google.com/spreadsheets/d/1DwWg3qdCRLI9k4rmDeMfH1PGv8XPoXsxTjZ88gvQpjA/edit?pli=1&gid=1994166085#gid=1994166085"
worksheet_name = "Agent Count"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

# Step 4: Clear existing data from the worksheet
gs_reader.clear_sheets(worksheet_name, start_row=1, start_column=1)

# Step 5: Write the new DataFrame to the worksheet (including headers)
gs_reader.set_df_in_sheets(worksheet_name, df, include_headers=True, row=1, col=1)